# 第〇步：开始之前 —— Jupyter 基础操作> ⚠️ 第一次用 Jupyter？花 3 分钟看完这一节。老手直接跳到 1.1。## Jupyter 是什么？Jupyter Notebook 是一个**在浏览器里写代码**的工具。最小单位叫 **Cell（单元格）**——可以是代码，也可以是笔记。## 三个必须记住的快捷键| 操作 | 快捷键 ||------|--------|| 运行当前 Cell | `Shift + Enter` || 新建 Cell | `Esc` 然后 `b` || 编辑 Cell | 直接点进去 |## 两个核心概念### ① `库/模块` —— 工具箱`import` 就是把「工具箱」拿出来。Python 自带一些，社区贡献了更多。之后用 `工具箱名.功能()` 来调用。```pythonimport pandas as pd   # 把 pandas 工具箱拿出来，给它起个小名叫 pd```如果提示 `ModuleNotFoundError`，说明工具箱还没装。在终端执行 `pip install 包名`。### ② `变量` —— 贴了标签的盒子```pythonprice = 1680.5    # 把 1680.5 装进盒子，贴上标签 "price"````=` 不是「等于」，是**「赋值」**。右边的东西装进左边名字的盒子里。### ③ `注释` —— 给未来自己的便条```pythonpe = 25  # ← 井号后面的内容 Python 会忽略```---

# 第一讲：Python 核心语法速通**学习目标**- 回顾列表、字典等核心数据结构- 掌握列表推导式 —— Python 最优雅的写法- 理解函数定义、lambda 和参数传递- 学会文件读写与异常处理- 为后续 NumPy / Pandas 量化分析扫清障碍> 💡 有基础的同学可以快速浏览，重点看 1.3（列表推导式）和 1.6（异常处理）。---

### 看看你的 Python 版本`sys` 是 Python 自带的系统工具箱，里面有 Python 版本信息。`f"..."` 是格式化字符串，花括号里的变量会被替换成它的值。

In [ ]:
import sysprint(f"Python 版本: {sys.version}")

## 1.1 数据类型速查Python 有四种复合数据类型。想象你在管理一个股票账户：| 类型 | 符号 | 特点 | 类比 ||------|------|------|------|| **列表** | `[]` | 有序、可修改 | 购物清单——可以加东西、改顺序 || **元组** | `()` | 有序、不可修改 | 身份证号——一旦确定就不能改 || **集合** | `{}` | 无序、不重复 | 行业分类——"银行"不会出现两次 || **字典** | `{key: value}` | 键值对查询 | 通讯录——通过名字查电话 |量化分析里几乎只用**列表**（存价格序列）和**字典**（存股票信息）。> ⚠️ **坑：列表索引从 0 开始！** `prices[0]` 是第一个，不是 `prices[1]`。负数索引从末尾往前：`prices[-1]` 是最后一个。### 字典取值：`[]` vs `.get()`- `dict['key']` —— 键不存在时直接报错崩溃- `dict.get('key', '默认值')` —— 键不存在时返回默认值，更安全量化代码里推荐用 `.get()`。

In [ ]:
prices = [10.5, 11.2, 10.8, 11.5]prices.append(12.0)print(f"列表: {prices}")print(f"  长度: {len(prices)}, 第2个: {prices[1]}, 最后: {prices[-1]}")date_tuple = (2024, 1, 15)print(f"\n元组: {date_tuple}, 年={date_tuple[0]}, 月={date_tuple[1]}")sectors = {"银行", "科技", "消费", "银行"}print(f"\n集合（自动去重）: {sectors}")stock_info = {"name": "茅台", "price": 1680.5, "pe": 35.2}print(f"\n字典: {stock_info}")print(f"  股票名: {stock_info['name']}")print(f"  PE: {stock_info.get('pe', '无数据')}")print(f"  市值: {stock_info.get('market_cap', '⚠ 缺数据')}")

## 1.2 条件判断与循环### 条件判断 `if / elif / else````pythonif 条件A:    做Xelif 条件B:      # "否则如果"else:            # "以上都不是"```**Python 用缩进判断代码块。** 同一层缩进的代码是一个整体。### Python 特有的「连续比较」```pythonif 0 < pe < 15:    # 等价于 pe > 0 and pe < 15，但更简洁```这在其他语言里写不了，Python 里可以。### `enumerate`：同时拿序号和值```pythonfor i, name in enumerate(stocks, start=1):    # i=1, name="茅台"; i=2, name="宁德"; ...```### `zip`：把多个列表「拉链」起来```pythonfor d, p in zip(dates, prices):    # d=dates[0], p=prices[0]; d=dates[1], p=prices[1]; ...```量化分析里经常对齐日期和价格。

In [ ]:
pe = 25if 0 < pe < 15:    label = "低估"elif 15 <= pe < 30:    label = "合理"else:    label = "高估"print(f"PE={pe} → {label}")stocks = ["茅台", "宁德", "比亚迪"]for i, name in enumerate(stocks, start=1):    print(f"  #{i}: {name}")dates = ["01-15", "01-16", "01-17"]prices = [1680, 1695, 1672]print("\n日期 vs 价格:")for d, p in zip(dates, prices):    print(f"  {d}: {p}")

## 1.3 列表推导式 —— Python 的「加速器」列表推导式用一行代码完成 **循环 + 筛选 + 变换**。```python[对元素的操作  for 元素 in 列表  if 条件]```### 为什么学它？1. 比传统 for 循环快（底层用 C 实现）2. 是 NumPy 向量化思想的铺垫 —— 习惯「对整列数据做同一件事」### 语法拆解```python[p / 100 for p in prices if p > 20] ↑操作     ↑循环变量  ↑被循环的列表    ↑过滤条件（可选）```### 收益率计算实战```python# zip(prices[:-1], prices[1:]) 把相邻两天配对daily_returns = [(b - a) / a for a, b in zip(prices[:-1], prices[1:])]````prices[:-1]` 去掉最后一天，`prices[1:]` 去掉第一天——配成 (day1,day2), (day2,day3)...

In [ ]:
prices = [10, 20, 30, 40, 50]returns_pct = []for p in prices:    returns_pct.append(p / 100)returns_pct_lc = [p / 100 for p in prices]print(f"传统: {returns_pct}")print(f"推导式: {returns_pct_lc}")print(f"是否相同: {returns_pct == returns_pct_lc}")

In [ ]:
prices = [10, 25, 8, 30, 15, 40, 5]adjusted = [p * 1.1 for p in prices if p > 20]print(f"原始: {prices}")print(f"筛选(>20)并上调10%: {adjusted}")stocks = ["茅台", "宁德"]dates = ["01-15", "01-16"]combinations = [(s, d) for s in stocks for d in dates]print(f"\n笛卡尔积: {combinations}")

In [ ]:
prices = [100, 102, 101, 105, 107]daily_returns = [(b - a) / a for a, b in zip(prices[:-1], prices[1:])]print(f"价格: {prices}")print(f"日收益率: {[f'{r:.2%}' for r in daily_returns]}")

## 1.4 函数与 Lambda### 函数 `def` —— 一段有名字的代码块```pythondef 函数名(参数):     # 参数 = 输入    做某事    return 结果       # return = 输出```类比榨汁机：放进橙子 → 流出橙汁。函数体内第一行用三个双引号括起来的文字叫 **docstring**，用来描述这个函数是干什么的。好的 docstring 让一个月后的自己还能看懂。### 默认参数```pythondef describe_stock(name, pe=None, sector="未知"):```调用时如果不传 `pe`，它的值就是 `None`（表示空）。如果没传 `sector`，默认值是 `"未知"`。### Lambda —— 一次性函数```pythonlambda 参数: 表达式```不需要起名字，用完就扔。常用于 `sorted(列表, key=lambda...)` 和 `map(lambda..., 列表)`。

In [ ]:
def calculate_return(start_price, end_price):    """计算简单收益率"""    return (end_price - start_price) / start_pricer = calculate_return(100, 108)print(f"100 → 108, 收益率: {r:.2%}")def describe_stock(name, pe=None, sector="未知"):    pe_str = f"PE={pe}" if pe is not None else "PE未提供"    return f"{name} | {sector} | {pe_str}"print(describe_stock("茅台", pe=35, sector="消费"))print(describe_stock("某次新股"))

In [ ]:
stocks = [    {"name": "茅台", "pe": 35},    {"name": "宁德", "pe": 45},    {"name": "工行", "pe": 5},]ranked = sorted(stocks, key=lambda s: s["pe"])print("按 PE 从小到大排序:")for s in ranked:    print(f"  {s['name']}: PE={s['pe']}")prices = [100, 200, 300]discounted = list(map(lambda p: p * 0.95, prices))print(f"\n原价: {prices}")print(f"打95折: {discounted}")

## 1.5 文件读写量化分析第一步几乎总是「从文件读数据」。### `with open(...) as f:` —— 最安全的写法- `"r"` = read（读）、`"w"` = write（写）- `encoding="utf-8"` 让中文不会乱码- `with` 语句结束后**自动关闭文件**，即使中途出错也不会泄漏> ⚠️ `"w"` 模式会**覆盖**已有文件。追加用 `"a"`（append）。### CSV 解析（不用 pandas 的情况下）```pythonheaders = f.readline().strip().split(",")  # 第一行是列名for line in f:    values = line.strip().split(",")        # 拆字段    record = dict(zip(headers, values))      # 列名+值 → 字典````.rstrip()` 去掉行末尾的换行符，`.strip()` 去掉两端空白。

In [ ]:
sample_data = """date,price2024-01-15,16802024-01-16,16952024-01-17,16722024-01-18,17002024-01-19,1715"""with open("sample_data.csv", "w", encoding="utf-8") as f:    f.write(sample_data)print("已写入 sample_data.csv")print("\n读取内容:")with open("sample_data.csv", "r", encoding="utf-8") as f:    for line in f:        print(f"  {line.rstrip()}")

In [ ]:
records = []with open("sample_data.csv", "r", encoding="utf-8") as f:    headers = f.readline().strip().split(",")    for line in f:        values = line.strip().split(",")        record = dict(zip(headers, values))        record["price"] = float(record["price"])        records.append(record)print("解析结果:")for r in records:    print(f"  {r['date']}: ¥{r['price']}")avg_price = sum(r["price"] for r in records) / len(records)print(f"\n平均价格: ¥{avg_price:.2f}")

## 1.6 异常处理量化代码经常要网络请求、文件读取、数据转换——这些都可能失败。`try/except` 的意思是：**试试看，如果出错，不要崩溃，走备用方案。**```pythontry:    可能出错的代码except 错误类型:    备用方案```### 为什么区分错误类型？- `ZeroDivisionError` → 除数为零 → 返回 NaN- `FileNotFoundError` → 文件不存在 → 提示用户- `TypeError` → 类型不对 → 返回 None不同错误用不同处理方式。如果只用 `except Exception`，你不知道为什么出错。### 常见坑- `float('nan')` 表示 Not a Number，让后续代码知道这里出过问题- `except` 不指定类型会吞掉所有错误，包括你不希望吞掉的（如 Ctrl+C）

In [ ]:
def safe_divide(a, b):    try:        return a / b    except ZeroDivisionError:        print(f"⚠ 警告: 除数为零 (a={a}, b={b})")        return float('nan')    except TypeError as e:        print(f"⚠ 类型错误: {e}")        return Noneprint(f"10/2 = {safe_divide(10, 2)}")print(f"10/0 = {safe_divide(10, 0)}")print(f"'a'/2 = {safe_divide('a', 2)}")

In [ ]:
def read_csv_if_exists(filename):    try:        with open(filename, "r", encoding="utf-8") as f:            return f.read()    except FileNotFoundError:        print(f"⚠ 文件 '{filename}' 不存在，返回空字符串")        return ""    except PermissionError:        print(f"⚠ 没有权限读取 '{filename}'")        return ""content = read_csv_if_exists("sample_data.csv")print(f"读取成功，{len(content)} 字符")missing = read_csv_if_exists("不存在的文件.csv")print(f"文件不存在: '{missing}'")

## 1.7 综合练习：简易持仓计算器把前面学的串起来：传入持仓和当前价格 → 输出总市值和盈亏。**用到的技能：** 字典 · 循环 · 条件判断 · Lambda 排序 · 格式化输出### 代码逻辑1. 遍历 `holdings` 字典，对每只股票：   - 取股数 × 成本价 = 成本   - 取股数 × 当前价 = 市值   - 市值 - 成本 = 盈亏2. 按收益率从高到低排序3. 打印格式化表格`f"{变量:>10.2f}"` 中的 `>10` 是右对齐占 10 格，`.2f` 是保留两位小数。

In [ ]:
def portfolio_summary(holdings, current_prices):    """计算持仓总市值和盈亏"""    total_value = 0    total_cost = 0    details = []    for name, info in holdings.items():        shares = info["shares"]        cost = info["cost"]        current = current_prices.get(name)        if current is None:            details.append(f"⚠ {name}: 缺少当前价格")            continue        market_value = shares * current        cost_value = shares * cost        pnl = market_value - cost_value        pnl_pct = (current - cost) / cost        total_value += market_value        total_cost += cost_value        details.append((name, shares, cost, current, pnl, pnl_pct))    details.sort(key=lambda x: x[5], reverse=True)    print(f"{'股票':<6} {'股数':>5} {'成本':>8} {'现价':>8} {'盈亏':>10} {'收益率':>8}")    print("-" * 55)    for d in details:        print(f"{d[0]:<6} {d[1]:>5} {d[2]:>8.2f} {d[3]:>8.2f} {d[4]:>+10.2f} {d[5]:>+7.2%}")    total_pnl = total_value - total_cost    total_pnl_pct = total_pnl / total_cost if total_cost > 0 else 0    print("-" * 55)    print(f"{'合计':<6} {'':>5} {'':>8} {'':>8} {total_pnl:>+10.2f} {total_pnl_pct:>+7.2%}")    return total_value, total_pnlmy_holdings = {    "茅台": {"shares": 100, "cost": 1600},    "宁德": {"shares": 500, "cost": 180},    "招商银行": {"shares": 1000, "cost": 35},}my_prices = {"茅台": 1700, "宁德": 175, "招商银行": 38}total, pnl = portfolio_summary(my_holdings, my_prices)print(f"\n总市值: ¥{total:,.2f}, 浮动盈亏: ¥{pnl:+,.2f}")

## 1.8 小结 & 自检清单| 技能 | ✓ ||------|---|| 理解 Jupyter Cell、`import`、变量的概念 | ☐ || 列表、字典、元组、集合的基本操作 | ☐ || 列表推导式（含条件筛选和嵌套） | ☐ || 函数定义、默认参数、`return` 的含义 | ☐ || Lambda 表达式（sorted、map） | ☐ || `with open` 文件读写（r/w 模式区别） | ☐ || `try/except` 异常处理（知道为什么区分错误类型） | ☐ || `zip` / `enumerate` / `sorted` 实用技巧 | ☐ |### 📝 你的笔记区在下面新建 Markdown Cell（`Esc` → `b` → `Esc` → `m`），写下：- 今天学到的最重要的 3 个概念- 1 个你还不理解的地方- 1 个你想明天尝试的练习---> 🎯 学完这一讲：能看懂 Python 量化代码中的基础语法，能写简单的数据处理脚本。**下一讲：** NumPy 数组运算与向量化操作。